## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Packages.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
import subprocess
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

Download repo and data files.

In [4]:
from src.data import configure_root

# Get root and insert to sys path
root = configure_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [ ]:
def sample_from_fasta(
        data_dir: Path,
        out_dir: Path,
        n_seqs: int,
        min_length: int,
        max_length: int,
        seed: int | None = None,
) -> pd.DataFrame:
    """
    Function to sample random sequences of a specified length range from .fa file, weigthed by chromsome length

    Args:
        fasta_file      : Path to .fa file containing genome
        chrom_size_file : Path to tab-separate text file containing sequence names and sizes
        n_seqs          : Number of sequences to sample
        min_length      : Minimum sequence length to sample
        max_length      : Maximum sequence length to sample
        seed            : Random seed for numpy
    
    Returns:
        df : Dataframe of sampled sequences with start, end, length, and chromosome ID
    """
    # Store paths to data
    fasta_file = data_dir / "hg38.fa"
    chrom_size_file = data_dir / "hg38.chrom.sizes"

    # Set random seed if specified
    rng = np.random.default_rng(seed)

    # Store file using pyfaidx
    f = Fasta(fasta_file)

    # Load the chromosome and sort
    size_df = pd.read_table(chrom_size_file, sep = "\t", index_col = 0, header = None)
    size_df.columns = ["length"]
    size_df["length"] = size_df["length"].astype(int)
    size_df = size_df.iloc[size_df.index.argsort()]

    # Check that the fasta chromsomes match the size df
    index = sorted(size_df.index.to_numpy())
    chromosomes = sorted(np.array(list(f.keys())))

    if index != chromosomes:
        raise KeyError("Fasta chromosomes IDs and length data IDs are incompatible")

    # Normalize to obtain sampling weights for each chromosome
    weights = size_df["length"].to_numpy() / size_df["length"].to_numpy().sum()

    # Initiate list to store sequences and metadata
    seqs = []
    chroms = []
    starts = []
    ends = []
    lengths = []

    # Counter so that rejected sequences don't count toward total count
    counter = 0

    # Number of sequences to sample
    while counter < n_seqs:

        # Weighted sampling of chromosome
        chrom = rng.choice(chromosomes, p = weights)
        length = int(size_df.loc[chrom, "length"])

        # Randomly sample a sequence
        seq_length = rng.integers(min_length, max_length + 1)
        start_idx = rng.integers(0, length - seq_length + 1)
        seq = f[chrom][start_idx:start_idx + seq_length]
        seq = str(seq).upper()

        # Split across "N" character if exists and take first substr
        seq = seq.split("N")[0]

        # Convert to uppercase, remove non-alpha and non-ACGT characters 
        seq = "".join(char for char in seq if char in "ACGT")
        new_length = len(seq)

        # Check that sequences is still above minimum length
        if new_length >= min_length:
            counter += 1 

            # Append sequence and all metadata
            seqs.append(seq)
            chroms.append(chrom)
            starts.append(start_idx)
            ends.append(start_idx + seq_length)
            lengths.append(new_length)

    # Construct dataframe
    df = pd.DataFrame({
        "sequence": seqs,
        "chromosome_id": chroms,
        "start": starts,
        "end": ends,
        "length": lengths
    })

    # Write to csv
    out_file = out_dir / "pretraining.csv.gz"
    df.to_csv(out_file, compression = "gzip", index = False)

    return df

In [ ]:
# Data directory
data_dir = root / "data"
out_dir = root

# Set parameters
n_seqs = 10**6
min_length = 500
max_length = 5000

# Get pretraining data
pretraining = sample_from_fasta(
    data_dir = data_dir,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length,
    seed = 111,
    out_dir = out_dir
)

## Tokenizer

Simple function to generate random samples to mimic fasta samples.

In [29]:
def generate_seqs(min_length, max_length, num_seqs, seed = None):  
    rng = np.random.default_rng(seed)
    seqs = []

    for i in range(num_seqs):
        length = rng.choice(np.arange(min_length, max_length))
        seq = [rng.choice(["A", "C", "G", "T"]) for i in range(length)]
        seq = "".join(seq)
        seqs.append(seq)
        
    return seqs

Train BPE tokenizer.

In [ ]:
from src.tokenize import BPETokenizer


## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Implement simple attention block.

https://medium.com/@heyamit10/implement-self-attention-and-cross-attention-in-pytorch-cfe17ab0b3ee

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        
        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

    def forward(self, x):

        # x[B, L, D]          
        # q[B, L, d_model], k[B, L, d_model], v[B, L, d_model]
        q = self.q_map(x)
        k = self.k_map(x)
        v = self.v_map(x)

        # a[B, L, L] (softmax over last dimension L from keys)
        a = torch.softmax(q @ k.transpose(-2, -1) / (self.d_model ** 0.5), dim = -1)

        # y[B, L, d_model]
        out = a @ v

        return out

Implement multi-head attention block.

In [ ]:
class MultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement sparse MultiHeadAttention.

In [ ]:
class SparseMultiHeadAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement transformer encoder block.

In [ ]:
class TransformerEncoder():
    def __init__(self):
        super().__init__()

Implement sparse attention transformer block.

In [ ]:
class SparseTransformerEncoder():
    def __init__(self):
        super().__init__()

Things that will be implemented:
- Padding to fixed max sequence length (512?)
- Embedding module
- Masked token prediction objective
- Sparse attention transformer
- 

Maybe pre-train on approx. 500k-1mil sequences?

Input to model should be a list of token IDs.

In [ ]:
class SparseGLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        
    ):
        super().__init__()

        self.embed = nn.Embedding

        # Transformer layers?
        
    def forward(self, x):

        return


class GLM(nn.Module):
    def __init(
        self,
        vocab_size
    ):
        super().__init__()



## Training

Implement masked language modeling training objective.
- Batching [B, L, D]
    - Token right padding should be done to the max length in the batch* (computational efficient)
- Attention mask to ignore padding tokens
- Train weights in batches

Reference: https://huggingface.co/learn/llm-course/en/chapter6/5